# 📘 RAG con múltiples índices usando RouterRetriever

Esta notebook muestra cómo construir dos índices separados con LlamaIndex y enrutar consultas utilizando `RouterRetriever`.

In [ ]:
!pip install -U llama-index langchain langchain-openai gdown --quiet

## 📁 Preparar documentos locales
Se descargan automáticamente los documentos para el RAG:
- Los CVs en una carpeta llamada `cv_licdia`
- El documento institucional como `documento_licdia.pdf`

A continuación se descargan los CV, en primer lugar:

In [ ]:
# Descargar todos los archivos PDF de la carpeta compartida (modo recursivo)
import gdown
import os

# Crear carpeta destino
os.makedirs("cv_licdia", exist_ok=True)

# ID de la carpeta compartida (sacada del link)
folder_id = "1ObKNPJ1DxZtIe5U1G29OmVlgJ6VW9LIe"

# Descargar todos los archivos de la carpeta al directorio
gdown.download_folder(id=folder_id, output="cv_licdia", quiet=False, use_cookies=False)

Retrieving folder contents


Processing file 1G2BKN5MsjdTC1Kz0j8ntuuNuOJA3tiTg Alvarez.pdf
Processing file 1zvghklDw8oyQE_ZdRpZAcWg3CLhIu2BR Cagnina.pdf
Processing file 1lye9yJRhDpCqpvy5TJ3g1om53G8Ngwvn Delfino.pdf
Processing file 1T0PY_V-oY8NttMmztPEk_S6670_LhQgh Errecalde.pdf
Processing file 1llbIQ74B8worywvdJs5i0O_8X1b2lUYb Fernandez.pdf
Processing file 13ixkIHsai42phGMKYmI92d2uotonO7GE Lanson.pdf
Processing file 1tbLBVWVSfmAbo23E_UkM26A_oBM1nfv7 Martinez.pdf
Processing file 1i8VGf-417MDsUtKN_FiEu6vMemM-90VB Matuk.pdf
Processing file 1x8hwKTifdLfOYLdcuGjM0E4PXsFDjp-U Penas-Steinhardt.pdf
Processing file 1KCfDxmEFfepb0-KViqysZypmtkUA5y8Q Petrocelli.pdf


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1G2BKN5MsjdTC1Kz0j8ntuuNuOJA3tiTg
To: /content/cv_licdia/Alvarez.pdf
100%|██████████| 96.9k/96.9k [00:00<00:00, 24.8MB/s]
Downloading...
From: https://drive.google.com/uc?id=1zvghklDw8oyQE_ZdRpZAcWg3CLhIu2BR
To: /content/cv_licdia/Cagnina.pdf
100%|██████████| 496k/496k [00:00<00:00, 25.8MB/s]
Downloading...
From: https://drive.google.com/uc?id=1lye9yJRhDpCqpvy5TJ3g1om53G8Ngwvn
To: /content/cv_licdia/Delfino.pdf
100%|██████████| 156k/156k [00:00<00:00, 19.2MB/s]
Downloading...
From: https://drive.google.com/uc?id=1T0PY_V-oY8NttMmztPEk_S6670_LhQgh
To: /content/cv_licdia/Errecalde.pdf
100%|██████████| 389k/389k [00:00<00:00, 27.5MB/s]
Downloading...
From: https://drive.google.com/uc?id=1llbIQ74B8worywvdJs5i0O_8X1b2lUYb
To: /content/cv_licdia/Fernandez.pdf
100%|██████████| 185k/185k [00:00<00:00, 7.98MB/s]
Downloading...
From: https://

['cv_licdia/Alvarez.pdf',
 'cv_licdia/Cagnina.pdf',
 'cv_licdia/Delfino.pdf',
 'cv_licdia/Errecalde.pdf',
 'cv_licdia/Fernandez.pdf',
 'cv_licdia/Lanson.pdf',
 'cv_licdia/Martinez.pdf',
 'cv_licdia/Matuk.pdf',
 'cv_licdia/Penas-Steinhardt.pdf',
 'cv_licdia/Petrocelli.pdf']

Se descarga la nota de solicitud de creación de LICDIA:

In [ ]:
# Descargar Google Docs como PDF
import requests

# ID del documento (lo que sigue a /d/ en el enlace)
document_id = "1CUhKhuuhlD0j-S6STDYkEfOfBv-LeYH9RwfYXy1B_rI"

# URL directa para exportar como PDF
url = f"https://docs.google.com/document/d/{document_id}/export?format=pdf"

# Nombre del archivo de salida
output_path = "documento_licdia.pdf"

# Realizar la descarga
response = requests.get(url)
if response.status_code == 200:
    with open(output_path, "wb") as f:
        f.write(response.content)
    print("✅ Documento descargado exitosamente como:", output_path)
else:
    print("❌ Error al descargar el documento. Verificá que sea público.")

✅ Documento descargado exitosamente como: documento_licdia.pdf


## ✂️ Creación de nodos

Se crean los nodos a partir de un splitter manual

In [ ]:
from llama_index.core import SimpleDirectoryReader
from llama_index.core.node_parser import SentenceSplitter

reader_cv = SimpleDirectoryReader("cv_licdia")
reader_lab = SimpleDirectoryReader(input_files=["documento_licdia.pdf"])

docs_cv = reader_cv.load_data()
docs_lab = reader_lab.load_data()

splitter = SentenceSplitter(chunk_size=512, chunk_overlap=50)
nodes_cv = splitter.get_nodes_from_documents(docs_cv)
nodes_lab = splitter.get_nodes_from_documents(docs_lab)

## 🔎 Crear índices y retrievers separados

Dado que vamos a usar OpenAI, cargamos el API-KEY:

In [ ]:
import os

os.environ["OPENAI_API_KEY"] = ""  # reemplazá por tu clave

Generamos los vectorstores:

In [ ]:
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.core import VectorStoreIndex

embed_model = OpenAIEmbedding()

index_cv = VectorStoreIndex(nodes_cv, embed_model=embed_model)
index_lab = VectorStoreIndex(nodes_lab, embed_model=embed_model)

## 🔀 RouterRetriever

Se utiliza un enrutador basado en LLM para aprovechar los dos índices definidos:

In [ ]:
from llama_index.llms.openai import OpenAI
from llama_index.core.selectors.llm_selectors import LLMSingleSelector
from llama_index.core.retrievers.router_retriever import RouterRetriever
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.tools import RetrieverTool


1️⃣ Se configura el LLM y se instancia el selector:

In [ ]:
llm = OpenAI(model="gpt-3.5-turbo", temperature=0)

selector = LLMSingleSelector.from_defaults(llm=llm)

2️⃣ Se definen los retrievers existentes a partir de los dos índices:

In [ ]:
retriever_cv = index_cv.as_retriever(similarity_top_k=3)
retriever_lab = index_lab.as_retriever(similarity_top_k=3)

3️⃣ Se envuelve cada retriever en un RetrieverTool:

In [ ]:
tool_cv = RetrieverTool.from_defaults(retriever=retriever_cv)
tool_lab = RetrieverTool.from_defaults(retriever=retriever_lab)

4️⃣ Se arma el RouterRetriever con las tool:

In [ ]:
router = RouterRetriever(
    selector=selector,
    retriever_tools=[tool_cv, tool_lab]
    )

## 💬 Consulta con enrute automático

In [ ]:
query_engine = RetrieverQueryEngine(retriever=router)

In [ ]:
response = query_engine.query("¿Çómo se impulsó la creación de LICDIA y quienes son sus investigadores? Explayate por favor.")

print(response)

La creación de LICDIA fue impulsada a través de la participación en proyectos de investigación relacionados con diversas áreas del conocimiento, como la ecología, la estadística y la probabilidad, la minería de textos y la inteligencia artificial. Los investigadores involucrados en LICDIA son expertos en diferentes campos, como Hugo DELFINO, Adonis David Nazareno GIORGI, Leticia Cecilia Cagnina, entre otros. Estos investigadores han liderado y participado en proyectos de investigación tanto a nivel nacional como internacional, abordando temáticas variadas y contribuyendo al avance del conocimiento en sus respectivas áreas de especialización.
